# Mark5.3 제안모델 — 여섯 요소 제거 · 단일 분류기
일반 특징의 9클래스 분류기를 학습·검증·추론에 공통으로 사용합니다. 특징 증류만 학습 보조로 남깁니다.
기본은 **선택 조건의 계획 확인**입니다. 아래 `RUN_TRAINING=True`로 학습과 val을 실행하고,
최종 test 평가 시 `EVALUATE_TEST=True`로 바꿉니다. 학습률 탐색과 과거 아블레이션 결과 복사는 하지 않습니다.


In [ ]:
import os, sys, subprocess, shutil, json, hashlib
from pathlib import Path
if "MARK_ROOT" not in os.environ:
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("MARK_ROOT", "/content/drive/MyDrive/NewProject/codes/experiments/Mark5.3")).resolve()
assert ROOT.is_dir(), ROOT
os.chdir(ROOT)
print("작업 폴더:", ROOT)


In [ ]:
# Colab의 GPU torch/torchvision은 유지합니다. 로컬 검증은 기존 환경을 사용합니다.
if "MARK_ROOT" not in os.environ:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "sentence-transformers==2.7.0", "transformers==4.46.3",
                    "huggingface-hub==0.36.2", "tokenizers==0.20.3"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "librosa==0.10.2.post1",
                    "soundfile==0.12.1", "pydub==0.25.1", "thop==0.1.1.post2209072238"], check=True)
import torch
print("torch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")


In [ ]:
# 기존 teacher가 사용하는 문장 인코더를 준비합니다.
if "MARK_ROOT" not in os.environ:
    hf_src = Path("/content/drive/MyDrive/NewProject/hf_cache/models--sentence-transformers--all-MiniLM-L6-v2")
    assert hf_src.is_dir(), f"문장 인코더 캐시가 없습니다: {hf_src}"
    hf_dst = Path.home() / ".cache/huggingface/hub" / hf_src.name
    shutil.copytree(hf_src, hf_dst, dirs_exist_ok=True)
    os.environ["HF_HUB_OFFLINE"] = "1"
    from sentence_transformers import SentenceTransformer
    SentenceTransformer("all-MiniLM-L6-v2")
# 실제 배포 파일과 teacher가 확인된 재료인지 검사합니다.
subprocess.run([sys.executable, "model/verify_release.py"], check=True)


In [ ]:
# 조건은 여기서만 지정합니다. 기본값은 한 조건, seed 1입니다.
LABELED = [2000]
UNLABELED = [2000]
SEEDS = [1]
LR = 1e-4
RUN_TRAINING = False
EVALUATE_TEST = False
RUN_PROFILE = False
BUNDLE = ("--linear_head --head_pretrained_hidden --hard_weight 1.0"
          " --others_from_reject 0.2 --neg_pl 0.1 0.3 --teacher_seg_mean"
          " --teacher_clean_view --bg_weight 0 --student_raw_wave --joint_forward")
COMMAND = [sys.executable, "-u", "model/run_experiments.py", "--mark_version", "mark5.2",
           "--labeled_ns", *map(str, LABELED), "--unlabeled_ns", *map(str, UNLABELED),
           "--seeds", *map(str, SEEDS), "--lr", str(LR), "--encoder_lr", str(LR),
           "--ignore_selected_lr", "--extra", BUNDLE]
subprocess.run(COMMAND + ["--skip_test", "--dry_run"], check=True)


In [ ]:
if RUN_TRAINING:
    subprocess.run(COMMAND + ["--skip_test"], check=True)
else:
    print("계획 확인만 완료. 선택 조건을 학습하려면 RUN_TRAINING=True로 바꿉니다.")


In [ ]:
# 이 셀은 학습 없이 선택 조건의 val 결과만 읽습니다.
import pandas as pd
sys.path.insert(0, str(ROOT / "model"))
from aggregate_results import collect
from run_experiments import _run_tag, _parse_runner_extra
_, _ab = _parse_runner_extra(BUNDLE)
SELECTED_TAGS = {_run_tag("mark5.2", s, u, labeled_n=None if l == 0 else l,
                          lr=LR, encoder_lr=LR, ablation_kw=_ab)
                 for l in LABELED for u in UNLABELED for s in SEEDS}
def show_selected(split):
    frame = collect("mark5.2", split)
    if frame.empty:
        print(f"완료된 새 {split} 결과가 없습니다."); return
    frame = frame[frame["run_tag"].isin(SELECTED_TAGS)].copy()
    if frame.empty:
        print(f"선택한 조건의 {split} 결과가 없습니다."); return
    out = ROOT / "plots" / f"single_selected_{split}.csv"
    frame.to_csv(out, index=False)
    display(frame)
    # 동일한 라벨 수·무라벨 수·seed의 기존 EfficientAT-PL 행만 함께 보여줍니다.
    base_path = ROOT.parent / "Baseline_EfficientAT_mn10" / "plots" / f"aggregate_raw_efficientat_mn10_{split}.csv"
    if base_path.is_file():
        baseline = pd.read_csv(base_path)
        baseline = baseline[(baseline["cond"] == "pl") & baseline["seed"].isin(SEEDS)
                            & baseline["labeled"].isin([2000 if l == 0 else l for l in LABELED])
                            & baseline["unlabeled"].isin(UNLABELED)]
        display(baseline)
    print("결과 저장:", out, "(한 seed 결과로 우위를 확정하지 않습니다.)")
show_selected("val")


In [ ]:
# 최종 test 평가: 기존 학습과 val은 체크포인트 해시가 맞으면 재사용합니다.
if EVALUATE_TEST:
    for tag in SELECTED_TAGS:
        assert (ROOT / "model" / f"student_model_{tag}.pth").is_file(), f"학습부터 완료하세요: {tag}"
    subprocess.run(COMMAND, check=True)
    show_selected("test")
else:
    print("최종 test 평가는 실행하지 않았습니다.")


In [ ]:
if RUN_PROFILE:
    subprocess.run([sys.executable, "model/profile_efficiency.py", "--mark_version", "mark5.2",
                    "--linear_head", "--head_pretrained_hidden"], check=True)


체크포인트는 `_SINGLE`, 평가 결과는 `_val_plain` / `_plain`을 사용합니다.
이전 체크포인트·보정 JSON·아블레이션 결과를 새 결과로 사용하지 않습니다.
본학습 성능 개선과 EfficientAT 초과 여부는 실제 완료한 결과로 판단해야 합니다.
